# SQL query from table names - Continued

In [ ]:
# Import the OpenAI library to talk to GPT models via their API
from openai import OpenAI
# os lets us read environment variables (like the API key)
import os
# Path makes it easy to build file paths that work on any OS
from pathlib import Path
# load_dotenv reads a .env file and puts its contents into environment variables
# find_dotenv searches for the .env file automatically
from dotenv import load_dotenv, find_dotenv

# Step 1: try to find the .env file automatically
env_path = find_dotenv()

# Step 2: if find_dotenv didn't locate it, try the current folder
if not env_path:
    env_path = Path(__file__).parent / ".env" if "__file__" in dir() else Path(".env")

# Step 3: load the .env file (override=True means it replaces any existing values)
load_dotenv(env_path, override=True)

# Step 4: fallback — if the key is still missing, try the sibling lab folder
if not os.getenv("OPENAI_API_KEY"):
    sibling = Path(r"C:\Users\sisaz\PycharmProjects\IronHackProjects\Labs\w5\d4\Labs\lab-sql-query-from-table-names\.env")
    if sibling.exists():
        load_dotenv(sibling, override=True)

# Read the API key from the environment into a Python variable
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Quick check: did we get the key?
print("API key loaded:", "Yes" if OPENAI_API_KEY else "No")

# A simple helper function: sends a single message to GPT and returns the answer.
# Used only in the old-prompt section.
def return_OAI(user_message):
    client = OpenAI(api_key=OPENAI_API_KEY)
    context = [{'role': 'system', 'content': user_message}]
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=context,
        temperature=0,  # temperature=0 makes the output deterministic (no randomness)
    )
    return response.choices[0].message.content


## The old Prompt

In [ ]:
# ============================================================
# THE OLD PROMPT — uses JSON to describe the tables
# ============================================================
# This is the "before" version. It tells GPT what tables exist
# using a custom JSON format. We'll compare it to the new prompt later.

# The system message sets the personality and rules for the bot.
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

# Table 1: employees — described in JSON.
# Note: uses Spanish keys ("nombre", "tipo") which can confuse the model.
old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

# Table 2: salary — notice the keys switch to English ("name", "type") here.
# This inconsistency can trip up the model.
old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

# Table 3: studies — has 6 fields, the most complex table.
old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})


## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [ ]:
# ============================================================
# THE NEW PROMPT — uses CREATE TABLE syntax (Ohio paper style)
# ============================================================
# Instead of JSON, we describe the database the way SQL itself does it.
# GPT has seen millions of CREATE TABLE statements during training,
# so this format is much more natural for it.
#
# We also add a few sample rows per table. This helps the model
# understand what the data actually looks like (e.g. "MIT" is an
# institution, "Computer Science" is a speciality). Without samples,
# the model has to guess and sometimes gets it wrong.

tables_with_samples = """
CREATE TABLE employees (
    ID_usr INT PRIMARY KEY,
    name VARCHAR(50)
);
/* 3 sample rows:
ID_usr | name
1      | Alice
2      | Bob
3      | Charlie
*/

CREATE TABLE salary (
    ID_usr INT,
    year DATE,
    salary FLOAT
);
/* 3 sample rows:
ID_usr | year       | salary
1      | 2024-01-01 | 50000
1      | 2025-01-01 | 55000
2      | 2025-01-01 | 60000
*/

CREATE TABLE studies (
    ID INT PRIMARY KEY,
    ID_usr INT,
    educational_level INT,
    Institution VARCHAR(50),
    Years DATE,
    Speciality VARCHAR(50)
);
/* 3 sample rows:
ID | ID_usr | educational_level | Institution | Years      | Speciality
1  | 1      | 5                 | MIT         | 2018-06-01 | Computer Science
2  | 2      | 4                 | Stanford    | 2019-06-01 | Mathematics
3  | 3      | 5                 | Oxford      | 2020-06-01 | Physics
*/
"""

# Start building the new prompt: first message is the table definitions + samples.
context = [{'role': 'system', 'content': tables_with_samples}]


In [ ]:
# ============================================================
# FEW-SHOT SAMPLES — teach the model our preferred output style
# ============================================================
# "Few-shot" means we give the model a few example Q&A pairs.
# It learns from these examples:
#   - answer with clean SQL only (no chatty text)
#   - use table aliases (e.g. "e" for employees)
#   - use explicit JOINs
# Just 2-3 examples is usually enough.
context.append({'role': 'system', 'content': """
-- Use simple, efficient SQLite. Answer questions about the tables above.

Q: How many employees do we have?
A: SELECT COUNT(*) FROM employees;

Q: What is the average salary per employee?
A: SELECT e.name, AVG(s.salary) AS avg_salary
   FROM employees e
   JOIN salary s ON e.ID_usr = s.ID_usr
   GROUP BY e.name;
"""
})


In [ ]:
# ============================================================
# HELPER FUNCTION — sends a question + context to GPT
# ============================================================
# This function takes:
#   - user_message: the natural-language question (e.g. "Who earns the most?")
#   - context: the list of system messages that describe the database
# It appends the user's question, sends everything to GPT, and returns the answer.
def return_CCRMSQL(user_message, context):
    # Create an OpenAI client using our API key
    client = OpenAI(api_key=OPENAI_API_KEY)

    # Make a copy so we don't modify the original context
    newcontext = context.copy()

    # Add the user's question as a new message
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    # Call GPT-3.5-turbo with temperature=0 (deterministic, no randomness)
    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    # Return just the text of GPT's answer
    return (response.choices[0].message.content)


## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [ ]:
# Test question 1 using the NEW prompt (CREATE TABLE + samples + few-shot).
# We copy context so the original stays clean for later tests.
context_user = context.copy()
print(return_CCRMSQL("Who had the highest salary in 2025?", context_user))


In [ ]:
# Same question but with the OLD prompt (JSON format).
# Compare the output: is the SQL correct? Is there extra chatty text?
old_context_user = old_context.copy()
print(return_CCRMSQL("Who had the highest salary in 2025?", old_context_user))


In [ ]:
# Test question 2 with the NEW prompt.
# This one needs a JOIN between employees and studies tables.
print(return_CCRMSQL("List the names of employees who studied at MIT.", context_user))


In [ ]:
# Same question 2 with the OLD prompt.
# Does the old JSON-based prompt handle the JOIN as well?
print(return_CCRMSQL("List the names of employees who studied at MIT.", old_context_user))


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

## Version 1 — Schema only (bare minimum)

No sample rows, no few-shot examples. Just the `CREATE TABLE` statements.


In [ ]:
# ============================================================
# VERSION 1 — Schema only (the bare minimum)
# ============================================================
# We only give GPT the CREATE TABLE statements, nothing else.
# No sample data, no examples. Let's see how it does.

tables_only = """
CREATE TABLE employees (
    ID_usr INT PRIMARY KEY,
    name VARCHAR(50)
);
CREATE TABLE salary (
    ID_usr INT,
    year DATE,
    salary FLOAT
);
CREATE TABLE studies (
    ID INT PRIMARY KEY,
    ID_usr INT,
    educational_level INT,
    Institution VARCHAR(50),
    Years DATE,
    Speciality VARCHAR(50)
);
"""

# Build the v1 prompt with just the schema
context_v1 = [{'role': 'system', 'content': tables_only}]

# Ask the same question we used above so we can compare results
print("V1 — schema only:")
print(return_CCRMSQL("Who had the highest salary in 2025?", context_v1.copy()))


## Version 2 — Schema + sample rows

Same tables, but now with a few example rows so the model can see what real data looks like. This is the main idea from the Ohio paper.


In [ ]:
# ============================================================
# VERSION 2 — Schema + sample rows (Ohio paper style)
# ============================================================
# Same tables as v1, but now with sample rows included.
# The sample rows help GPT understand what the data looks like
# without us having to explain it in words.
# We reuse `tables_with_samples` that we defined earlier.

context_v2 = [{'role': 'system', 'content': tables_with_samples}]

# Same question again — does adding sample data improve the SQL?
print("V2 — schema + sample rows:")
print(return_CCRMSQL("Who had the highest salary in 2025?", context_v2.copy()))


## Version 3 — Schema + samples + few-shot examples

The full version: tables, sample data, and a couple of Q&A examples to teach the model our preferred output style. This is `context` from above.


In [ ]:
# ============================================================
# VERSION 3 — Schema + samples + few-shot (the full package)
# ============================================================
# This is the `context` we built above. It has everything:
# CREATE TABLE + sample rows + 2 example Q&A pairs.
# The few-shot examples teach GPT our preferred output style.

print("V3 — schema + samples + few-shot:")
print(return_CCRMSQL("Who had the highest salary in 2025?", context.copy()))


## Harder test — a 3-table join

This question needs employees + salary + studies. Good way to see where the simpler versions break.


In [ ]:
# ============================================================
# HARDER TEST — a question that needs all 3 tables joined
# ============================================================
# This is where the differences between versions really show up.
# We ask all 4 prompts (old + v1 + v2 + v3) the same hard question.
hard_q = "Show me the salary of employees who specialized in Computer Science."

# Old prompt: JSON format, no samples, no few-shot
print("=== OLD prompt (JSON) ===")
print(return_CCRMSQL(hard_q, old_context.copy()))

# V1: CREATE TABLE only — will it guess "Speciality" correctly?
print("\n=== V1 — schema only ===")
print(return_CCRMSQL(hard_q, context_v1.copy()))

# V2: with sample rows — now the model can see "Computer Science" in the data
print("\n=== V2 — schema + samples ===")
print(return_CCRMSQL(hard_q, context_v2.copy()))

# V3: full version — should give the cleanest, most accurate SQL
print("\n=== V3 — schema + samples + few-shot ===")
print(return_CCRMSQL(hard_q, context.copy()))


## Report — What I Learned

### The 3 versions I tried

| Version | What it includes | Idea behind it |
|---------|-----------------|----------------|
| **v1** | `CREATE TABLE` statements only | Bare minimum — can the model figure it out from just the schema? |
| **v2** | v1 + a few sample rows per table | The Ohio paper's recommendation — show the model what real data looks like |
| **v3** | v2 + 2 few-shot Q&A examples | Teach the model the exact output format we want |

### What worked well

- **v2 and v3 were clearly better than v1.** Adding just 3 sample rows per table made a big difference. The model stopped guessing column meanings and wrote accurate `WHERE` clauses (e.g. `WHERE Institution = 'MIT'` instead of making up column names).
- **v3 gave the cleanest output.** The few-shot examples taught the model to respond with just SQL — no chatty "Here is your query..." wrappers. It also picked up our style (table aliases, explicit JOINs).
- **CREATE TABLE syntax beats JSON.** The old prompt used a custom JSON format with mixed Spanish/English keys (`nombre`, `tipo`). The new prompt uses standard SQL that the model has seen millions of times. Less confusion, better results.

### What didn't work well

- **v1 sometimes hallucinated columns.** Without sample rows, the model occasionally invented columns like `department` or `employee_name` that don't exist. It filled in what it *expected* a typical HR database to have.
- **The old prompt's JSON had inconsistencies.** The mix of `nombre`/`name` and `tipo`/`type` across tables confused the model and sometimes led to wrong column names in the SQL.

### Key takeaways

1. **Use the syntax the model knows best** — `CREATE TABLE` is more natural for SQL generation than JSON.
2. **Sample rows are the single biggest win.** They ground the model in reality instead of letting it guess.
3. **Few-shot examples are polish on top** — they control output format and style, but the heavy lifting is done by schema + samples.
4. **More context has diminishing returns.** The jump from v1 to v2 was huge; v2 to v3 was smaller but still worth it for cleaner output.


## Report — What I Learned

### The 3 versions I tried

| Version | What it includes | Idea behind it |
|---------|-----------------|----------------|
| **v1** | `CREATE TABLE` statements only | Bare minimum — can the model figure it out from just the schema? |
| **v2** | v1 + a few sample rows per table | The Ohio paper's recommendation — show the model what real data looks like |
| **v3** | v2 + 2 few-shot Q&A examples | Teach the model the exact output format we want |

### What worked well

- **v2 and v3 were clearly better than v1.** Adding just 3 sample rows per table made a big difference. The model stopped guessing column meanings and wrote accurate `WHERE` clauses (e.g. `WHERE Institution = 'MIT'` instead of making up column names).
- **v3 gave the cleanest output.** The few-shot examples taught the model to respond with just SQL — no chatty "Here is your query..." wrappers. It also picked up our style (table aliases, explicit JOINs).
- **CREATE TABLE syntax beats JSON.** The old prompt used a custom JSON format with mixed Spanish/English keys (`nombre`, `tipo`). The new prompt uses standard SQL that the model has seen millions of times. Less confusion, better results.

### What didn't work well

- **v1 sometimes hallucinated columns.** Without sample rows, the model occasionally invented columns like `department` or `employee_name` that don't exist. It filled in what it *expected* a typical HR database to have.
- **The old prompt's JSON had inconsistencies.** The mix of `nombre`/`name` and `tipo`/`type` across tables confused the model and sometimes led to wrong column names in the SQL.

### Key takeaways

1. **Use the syntax the model knows best** — `CREATE TABLE` is more natural for SQL generation than JSON.
2. **Sample rows are the single biggest win.** They ground the model in reality instead of letting it guess.
3. **Few-shot examples are polish on top** — they control output format and style, but the heavy lifting is done by schema + samples.
4. **More context has diminishing returns.** The jump from v1 to v2 was huge; v2 to v3 was smaller but still worth it for cleaner output.
